## SQL - Major SQL commands - sqlite recap and DBeaver and mysql
### BIOINF 575

#### Guidelines for database design:

* Normalization is the process of creating or re-arranging data relationships so that it will be easy to store and retrieve data efficiently.  Data is normalized to achieve the following goals: 
    * Eliminate data redundancies and save space 
    * Make it easier to change data 
    * Simplify the enforcement of referential integrity constraints 
    * Produce a design that is a 'good' representation of the real world (one that is intuitively easy to understand and a good base for further growth)

    * Make it easier to change data by avoiding to provide multiple values separated by commas in a column
    * All columns in a table should depend on the primary key, all extra related information should be in other tables linked by foreign keys

https://support.microsoft.com/en-us/help/283878/description-of-the-database-normalization-basics

##### RESOURCES
https://dev.mysql.com/doc/refman/8.0/en/         
https://www.w3schools.com/mysql/mysql_create_table.asp          
https://www.mysqltutorial.org/mysql-sample-database.aspx       
https://www.tutorialspoint.com/mysql/index.htm    
https://realpython.com/python-mysql/




https://dev.mysql.com/doc/refman/8.0/en/examples.html

#### Connect to the database

```mysql your-database-name```

#### Create a table and add data, then select data from the table

```sql
CREATE TABLE shop (
    article INT UNSIGNED  DEFAULT '0000' NOT NULL,
    dealer  CHAR(20)      DEFAULT ''     NOT NULL,
    price   DECIMAL(16,2) DEFAULT '0.00' NOT NULL,
    PRIMARY KEY(article, dealer));

INSERT INTO shop VALUES
    (1,'A',3.45),(1,'B',3.99),(2,'A',10.99),(3,'B',1.45),
    (3,'C',1.69),(3,'D',1.25),(4,'D',19.95);
    
SELECT * FROM shop ORDER BY article;

```


| article   | dealer   | price   |
| --------- |--------  | ------- |
|       1   | A        |  3.45   |
|       1   | B        |  3.99   |
|       2   | A        | 10.99   |
|       3   | B        |  1.45   |
|       3   | C        |  1.69   |
|       3   | D        |  1.25   |
|       4   | D        | 19.95   |

#### Examples of common querries

https://dev.mysql.com/doc/refman/8.0/en/examples.html

In [ ]:
### install pymysql using the following command in a terminal:
! pip install pymysql

In [ ]:
from pymysql import connect


In [ ]:
from pymysql import connect

#Create a connection object
conn =connect(host='ensembldb.ensembl.org', user='anonymous',  port=5306, db = "saccharomyces_cerevisiae_core_94_4")


In [ ]:
# May need a password: password = 'password'
# Can request specific database: db = 'database'
# Get a cursor – it sends SQL statements and receives responses
cursor = conn.cursor()
sql = "show tables"
cursor.execute(sql)
for (table_name,) in cursor: 
    print(table_name) 

#Do your queries, work with responses



In [ ]:
import pandas as pd
select_genes = """
SELECT * 
FROM gene
LIMIT 5;
"""

cursor.execute(select_genes)
header = [t[0] for t in cursor.description]
df = pd.DataFrame(cursor.fetchall(), columns = header)
df

In [ ]:
cursor.description

In [ ]:
select_genes = """
SELECT gene_id, biotype, is_current
FROM gene
LIMIT 20;
"""

cursor.execute(select_genes)
cursor.fetchall()

In [ ]:
cursor.description

In [ ]:
select_dnaseq = """
SELECT * 
FROM seq_region
LIMIT 20;
"""

cursor.execute(select_dnaseq)
cursor.fetchall()

In [ ]:
cursor.description

In [ ]:
select_dnaseq_syn = """
SELECT * 
FROM seq_region_synonym
LIMIT 20;
"""

cursor.execute(select_dnaseq_syn)
cursor.fetchall()

In [ ]:
cursor.description

In [ ]:
# Clean up - do this when done with the database
cursor.close()
conn.close()

### Using custom objects together with SQL
### sqlalchemy

#### We will follow the tutorial here:

https://www.sqlalchemy.org/library.html#talks

https://github.com/zzzeek/sqla_tutorial/blob/main/slides/04_orm_basic.py

Another resource:    
https://docs.sqlalchemy.org/en/14/orm/tutorial.html#version-check

In [ ]:
! pip install SQLAlchemy

In [ ]:
import sqlalchemy

In [ ]:
sqlalchemy.__version__ 

In [ ]:
# code from:
# https://www.sqlalchemy.org/library.html#talks
# https://github.com/zzzeek/sqla_tutorial/blob/main/slides/04_orm_basic.py

In [ ]:
### slide::
### title:: Object Relational Mapping
# SQLAlchemy mappings in 1.4 / 2.0 start with a central object
# known as the *registry*

from sqlalchemy.orm import registry


mapper_registry = registry()

### slide::
# Using the registry, we can map classes in various ways, below illustrated
# using its "mapped" decorator.
# In this form, we arrange class attributes in terms of Column objects
# to be mapped to a Table, which is named based on an attribute
# "__tablename__"

from sqlalchemy import Column, Integer, String


@mapper_registry.mapped
class User:
    __tablename__ = "user_account"

    id = Column(Integer, primary_key=True)
    username = Column(String)
    fullname = Column(String)

    def __repr__(self):
        return "<User(%r, %r)>" % (self.username, self.fullname)


### slide::
# the User class now has a Table object associated with it.

User.__table__

### slide:: i
# The Mapper object mediates the relationship between User
# and the "user" Table object.  This mapper object is generally behind
# the scenes.

User.__mapper__

### slide::
# User has a default constructor, accepting field names
# as arguments.

spongebob = User(username="spongebob", fullname="Spongebob Squarepants")
spongebob

### slide::
# Attributes which we didn't set, such as the "id", are displayed as
# None when we access them

repr(spongebob.id)


In [ ]:
spongebob

In [ ]:
### slide:: p
# Using our registry, we can create a database schema for this class using
# a MetaData object that is part of the registry.

from sqlalchemy import create_engine

engine = create_engine("sqlite:///test.sqlite")
with engine.begin() as connection:
    mapper_registry.metadata.create_all(connection)

### slide::
# To persist and load User objects from the database, we
# use a Session object, illustrated here from a factory called
# sessionmaker.  The Session object makes use of a connection
# factory (i.e. an Engine) and will handle the job of connecting,
# committing, and releasing connections to this engine.

from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine, future=True)

session = Session()

### slide::
# new objects are placed into the Session using add().
session.add(spongebob)



In [ ]:
### slide:: i
# This did not yet modify the database, however the object is now known as
# **pending**.  We can see the "pending" objects by looking at the session.new
# attribute.
session.new

In [ ]:
### slide:: p
# We can now query for this **pending** row, by emitting a SELECT statement
# that will refer to "User" entities.   This will first **autoflush**
# the pending changes, then SELECT the row we requested.

from sqlalchemy import select

select_statement = select(User).filter_by(username="spongebob")
result = session.execute(select_statement)
result

In [ ]:

### slide:: i
# We can get the data back from the result, in this case using the
# .scalar() method which will return the first column of the first row.
also_spongebob = result.scalar()
also_spongebob

In [ ]:

### slide::
# the User object we've inserted now has a value for ".id"
spongebob.id

In [ ]:
### slide:: i
# the Session maintains a *unique* object per identity.
# so "spongebob" and "also_spongebob" are the *same* object

spongebob is also_spongebob

In [ ]:
### slide:: i
# this is known as the **identity map**, and we can look at it on
# the Session.

session.identity_map.items()

In [ ]:
### slide::
### title:: Making Changes
# Add more objects to be pending for flush.

session.add_all(
    [
        User(username="patrick", fullname="Patrick Star"),
        User(username="sandy", fullname="Sandy Cheeks"),
    ]
)

In [ ]:


### slide:: i
# modify "spongebob" - the object is now marked as *dirty*.

spongebob.fullname = "Spongebob Jones"

### slide::
# the Session can tell us which objects are dirty...

session.dirty

In [ ]:

### slide:: i
# and can also tell us which objects are pending...

session.new

In [ ]:
### slide:: p i
# The whole transaction is committed.  Commit always triggers
# a final flush of remaining changes.

session.commit()

In [ ]:
### slide:: p
# After a commit, theres no transaction.  The Session
# *invalidates* all data, so that accessing them will automatically
# start a *new* transaction and re-load from the database.  This is
# our first example of the ORM *lazy loading* pattern.

spongebob.fullname

In [ ]:
### slide::
### title:: rolling back changes
# Make another "dirty" change, and another "pending" change,
# that we might change our minds about.

spongebob.username = "Spongy"
fake_user = User(username="fakeuser", fullname="Invalid")
session.add(fake_user)

In [ ]:
### slide:: p
# run a query, our changes are flushed; results come back.

result = session.execute(
    select(User).where(User.username.in_(["Spongy", "fakeuser"]))
)
result.all()

In [ ]:

### slide::
# But we're inside of a transaction.  Roll it back.
session.rollback()

In [ ]:

### slide:: p
# Again, the transaction is over, objects are expired.
# Accessing an attribute refreshes the object and the "Spongy" username is gone
spongebob.username

In [ ]:

### slide::
# "fake_user" has been evicted from the session.
fake_user in session

In [ ]:
### slide:: pi
# and the data is gone from the database too.

result = session.execute(
    select(User).where(User.username.in_(["spongebob", "fakeuser"]))
)
result.all()

In [ ]:

### slide::
### title:: ORM Querying
# The attributes on our mapped class act like Column objects, and
# produce SQL expressions.

print(User.username == "spongebob")


In [ ]:

### slide::
# When ORM-specific expressions are used with select(), the Select construct
# itself takes on ORM-enabled features, the most basic of which is that
# it can discern between selecting from *columns* vs *entities*.  Below,
# the SELECT is to return rows that contain a single element, which would
# be an instance of User.   This is translated from the actual SELECT
# sent to the database that SELECTs for the individual columns of the
# User entity.

query = (
    select(User).where(User.username == "spongebob").order_by(User.id)
)

### slide:: ip
# the rows we get back from Session.execute() then contain User objects
# as the first element in each row.
result = session.execute(query)

for row in result:
    print(row)

In [ ]:

### slide:: p
# As it is typically convenient for rows that only have a single element
# to be delivered as the element alone, we can use the .scalars() method
# of Result as we did earlier to return just the first column of each row

result = session.execute(query)
for user_obj in result.scalars():
    print(user_obj)

### slide:: pi
# we can also qualify the rows we want to get back with methods like
# .one()

result = session.execute(query)

user_obj = result.scalars().one()
print(user_obj)

In [ ]:

### slide:: p
# An ORM query can make use of any combination of columns and entities.
# To request the fields of User separately, we name them separately in the
# columns clause

query = select(User.username, User.fullname)
result = session.execute(query)
for row in result:
    print(f"{row.username} {row.fullname}")

### slide:: p
# as well as combinations of "entities" and columns
query = select(User, User.username)
result = session.execute(query)
for row in result:
    print(f"{row.User.id} {row.User.fullname} {row.username}")


In [ ]:





### slide:: p
# the WHERE clause is either by filter_by(), which is convenient

for (username, ) in session.execute(
    select(User.username).filter_by(
        fullname="Spongebob Jones"
    )
):
    print(username)

### slide:: pi
# or where() for more explicitness

from sqlalchemy import or_

for (user, ) in (
    session.execute(
        select(User)
        .where(User.username == "spongebob")
        .where(or_(User.fullname == "Spongebob Jones", User.id < 5))
    )
):
    print(user)



In [ ]:
session.close_all?

In [ ]:
session.close?

In [ ]:
session.close()
engine.dispose()